# IPL Integrated Model — 2008 to 2026

```

## Section 1 — Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, joblib
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

PATH_2025 = r"C:\Users\Sravanth.claaps\IPL-project\Backend\data\2008-2025 ipl dataset"  
PATH_2026 = r"C:\Users\Sravanth.claaps\IPL-project\Backend\data\2026 dataset"                  
SAVE_DIR  = r"C:\Users\Sravanth.claaps\Downloads"                       

TEAM_COLORS = {
    "Mumbai Indians"              : "#004BA0",
    "Chennai Super Kings"         : "#F7A721",
    "Royal Challengers Bangalore" : "#EC1C24",
    "Kolkata Knight Riders"       : "#3A225D",
    "Delhi Capitals"              : "#17449B",
    "Punjab Kings"                : "#ED1B24",
    "Rajasthan Royals"            : "#E91E8C",
    "Sunrisers Hyderabad"         : "#FF6600",
    "Gujarat Titans"              : "#1C4899",
    "Lucknow Super Giants"        : "#00A3E0",
}

TEAM_MAP = {
    "Delhi Daredevils"        : "Delhi Capitals",
    "Kings XI Punjab"         : "Punjab Kings",
    "Deccan Chargers"         : "Sunrisers Hyderabad",
    "Gujarat Lions"           : "Gujarat Titans",
    "Rising Pune Supergiant"  : "Chennai Super Kings",
    "Pune Warriors India"     : "Chennai Super Kings",
    "Kochi Tuskers Kerala"    : "Chennai Super Kings",
}

# 2026 uses short names — expand them to full names
SHORT_TO_FULL = {
    "MI"  : "Mumbai Indians",
    "CSK" : "Chennai Super Kings",
    "RCB" : "Royal Challengers Bangalore",
    "KKR" : "Kolkata Knight Riders",
    "DC"  : "Delhi Capitals",
    "PBKS": "Punjab Kings",
    "RR"  : "Rajasthan Royals",
    "SRH" : "Sunrisers Hyderabad",
    "GT"  : "Gujarat Titans",
    "LSG" : "Lucknow Super Giants",
    "PK"  : "Punjab Kings",
}
FULL_TO_SHORT = {v: k for k, v in SHORT_TO_FULL.items()}

print(" Setup complete")

: 

## Section 2 — Load 2008–2025 Data (Your Original Dataset)

In [ ]:
matches_old   = pd.read_csv(os.path.join(PATH_2025, 'matches.csv'))
deliveries_old = pd.read_csv(os.path.join(PATH_2025, 'deliveries.csv'))

# Normalize team names
for col in ['team1', 'team2', 'toss_winner', 'winner']:
    if col in matches_old.columns:
        matches_old[col] = matches_old[col].replace(TEAM_MAP)

for col in ['batting_team', 'bowling_team']:
    if col in deliveries_old.columns:
        deliveries_old[col] = deliveries_old[col].replace(TEAM_MAP)

# Parse dates
matches_old['date'] = pd.to_datetime(matches_old['date'], errors='coerce')

print(f" 2008-2025 matches loaded  : {len(matches_old):,} rows")
print(f" 2008-2025 deliveries loaded: {len(deliveries_old):,} rows")
print(f"   Seasons covered: {sorted(matches_old['season'].unique())}")
print(f"   Columns: {list(matches_old.columns)}")

## Section 3 — Load & Normalize 2026 Data

In [ ]:
matches_26_raw    = pd.read_csv(os.path.join(PATH_2026, 'matches.csv'))
deliveries_26_raw = pd.read_csv(os.path.join(PATH_2026, 'deliveries.csv'))
batting_26        = pd.read_csv(os.path.join(PATH_2026, 'batting_stats.csv'))
bowling_26        = pd.read_csv(os.path.join(PATH_2026, 'bowling_stats.csv'))

def normalize_2026_matches(df):
    df = df.copy()

    df = df[df['match_result'] == 'completed'].reset_index(drop=True)

    for col in ['team1', 'team2', 'toss_winner', 'match_winner']:
        df[col] = df[col].map(lambda x: SHORT_TO_FULL.get(str(x).strip(), x))

    df = df.rename(columns={
        'first_ings_score'  : 'first_innings_score',
        'first_ings_wkts'   : 'first_innings_wickets',
        'second_ings_score' : 'second_innings_score',
        'second_ings_wkts'  : 'second_innings_wickets',
        'match_result'      : 'result',
        'match_winner'      : 'winner',
    })

    def get_win_by(row):
        if pd.notna(row['wb_wickets']) and row['wb_wickets'] > 0:
            return 'wickets'
        elif pd.notna(row['wb_runs']) and row['wb_runs'] > 0:
            return 'runs'
        else:
            return 'wickets' 

    df['win_by'] = df.apply(get_win_by, axis=1)

    df['toss_decision'] = df['toss_decision'].str.lower().map(
        lambda x: 'field' if x in ['bowl', 'field'] else 'bat'
    )

    df['season']       = 2026
    df['result']       = 'normal'
    df['is_day_night'] = True

    df['first_innings_overs']  = 20.0
    df['second_innings_overs'] = df.apply(
        lambda r: round((120 - float(r['balls_left'])) / 6, 1)
        if pd.notna(r.get('balls_left')) else 20.0, axis=1
    )

    df['date'] = pd.to_datetime(df['date'], format='%B %d, %Y', errors='coerce')

    return df

def normalize_2026_deliveries(df):
    df = df.copy()

    for col in ['batting_team', 'bowling_team']:
        df[col] = df[col].map(lambda x: SHORT_TO_FULL.get(str(x).strip(), x))

    df = df.rename(columns={
        'match_no'        : 'match_id',
        'runs_of_bat'     : 'batsman_runs',
        'player_dismissed': 'player_dismissed',
        'wicket_type'     : 'dismissal_kind',
    })

    df['total_runs'] = df['batsman_runs'] + df['extras']
    df['season']     = 2026

    return df


matches_26    = normalize_2026_matches(matches_26_raw)
deliveries_26 = normalize_2026_deliveries(deliveries_26_raw)

print(f" 2026 matches normalized   : {len(matches_26)} completed matches")
print(f" 2026 deliveries normalized: {len(deliveries_26):,} balls")
print(f"\n   Teams in 2026: {sorted(matches_26['winner'].dropna().unique())}")
print(f"\n   Column check (first 5 rows):")
matches_26[['date','team1','team2','toss_winner','toss_decision',
            'first_innings_score','second_innings_score','winner','win_by','season']].head(5)

## Section 4 — Merge into One Unified 2008–2026 Dataset

In [ ]:
cols_old = set(matches_old.columns)
cols_new = set(matches_26.columns)
common   = sorted(cols_old & cols_new)
only_old = sorted(cols_old - cols_new)
only_new = sorted(cols_new - cols_old)

print(f"Common columns ({len(common)}): {common}")
print(f"\nOnly in 2008-2025 ({len(only_old)}): {only_old}")
print(f"Only in 2026 ({len(only_new)}):      {only_new}")

CORE_COLS = [
    'date', 'season', 'venue', 'team1', 'team2',
    'toss_winner', 'toss_decision',
    'first_innings_score', 'first_innings_wickets',
    'first_innings_overs',
    'second_innings_score', 'second_innings_wickets',
    'second_innings_overs',
    'result', 'winner', 'win_by', 'is_day_night'
]

old_cols_avail = [c for c in CORE_COLS if c in matches_old.columns]
new_cols_avail = [c for c in CORE_COLS if c in matches_26.columns]

df_old = matches_old[old_cols_avail].copy()
df_new = matches_26[new_cols_avail].copy()

all_core = sorted(set(old_cols_avail) | set(new_cols_avail))
for col in all_core:
    if col not in df_old.columns:
        df_old[col] = np.nan
    if col not in df_new.columns:
        df_new[col] = np.nan


matches_all = pd.concat([df_old, df_new], ignore_index=True)
matches_all = matches_all.sort_values('date').reset_index(drop=True)

print(f"\n MERGED DATASET")
print(f"   2008-2025 matches : {len(df_old):,}")
print(f"   2026 matches      : {len(df_new):,}")
print(f"   TOTAL             : {len(matches_all):,}")
print(f"   Seasons covered   : {sorted(matches_all['season'].dropna().unique())}")
print(f"   Date range        : {matches_all['date'].min().date()} → {matches_all['date'].max().date()}")

per_season = matches_all.groupby('season').size()
print(f"\n   Matches per season:\n{per_season.to_string()}")

## Section 5 — Merge Deliveries Dataset

In [ ]:
del_common = sorted(set(deliveries_old.columns) & set(deliveries_26.columns))
print(f"Common delivery columns: {del_common}")

del_old = deliveries_old[[c for c in del_common if c in deliveries_old.columns]].copy()
del_new = deliveries_26[[c for c in del_common if c in deliveries_26.columns]].copy()

del_old['season'] = del_old.get('season', pd.Series(dtype=int))
del_new['season'] = 2026

deliveries_all = pd.concat([del_old, del_new], ignore_index=True)

print(f"\n Merged deliveries:")
print(f"   2008-2025: {len(del_old):,} balls")
print(f"   2026     : {len(del_new):,} balls")
print(f"   TOTAL    : {len(deliveries_all):,} balls")

## Section 6 — EDA: Compare 2025 vs 2026 Season

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("IPL 2008–2026 — Integrated Dataset Analysis", fontsize=16, fontweight='bold')

m_normal = matches_all[matches_all['result'] == 'normal'].copy()

#Matches per season (all years)
m_season = m_normal.groupby('season').size()
bar_colors = ['#EC1C24' if s == 2026 else '#004BA0' for s in m_season.index]
axes[0,0].bar(m_season.index, m_season.values, color=bar_colors)
axes[0,0].set_title("Matches Per Season (2026 in red)")
axes[0,0].set_xlabel("Season"); axes[0,0].set_ylabel("Matches")
for x, y in zip(m_season.index, m_season.values):
    axes[0,0].text(x, y+0.3, str(y), ha='center', fontsize=7)

#Avg 1st innings score per season
avg_score = m_normal.groupby('season')['first_innings_score'].mean()
axes[0,1].plot(avg_score.index, avg_score.values, marker='o',
               color='#F7A721', linewidth=2.5, markersize=7)
axes[0,1].fill_between(avg_score.index, avg_score.values, alpha=0.15, color='#F7A721')
axes[0,1].axvline(x=2026, color='red', linestyle='--', alpha=0.6, label='2026')
axes[0,1].set_title("Avg 1st Innings Score per Season")
axes[0,1].set_xlabel("Season"); axes[0,1].set_ylabel("Runs")
axes[0,1].legend()

#Toss decision impact (2025 vs 2026)
for ax, season, color in zip([None, axes[0,2]], [2025, 2026], ['#004BA0', '#EC1C24']):
    subset = m_normal[m_normal['season'] == season].copy()
    subset['toss_won'] = (subset['toss_winner'] == subset['winner'])
    td = subset.groupby('toss_decision')['toss_won'].mean() * 100
    axes[0,2].bar([f"{season}\n{d}" for d in td.index], td.values,
                  color=color, alpha=0.8, label=str(season))
axes[0,2].set_title("Toss Decision → Win % (2025 vs 2026)")
axes[0,2].set_ylabel("Win %"); axes[0,2].legend()

#Win method distribution: runs vs wickets
win_method = m_normal.groupby(['season', 'win_by']).size().unstack(fill_value=0)
win_method_pct = win_method.div(win_method.sum(axis=1), axis=0) * 100
win_method_pct.plot(kind='bar', ax=axes[1,0], color=['#004BA0', '#F7A721'],
                    alpha=0.85, edgecolor='white')
axes[1,0].set_title("Win Method % per Season")
axes[1,0].set_xlabel("Season"); axes[1,0].set_ylabel("%")
axes[1,0].legend(['By Runs', 'By Wickets'], loc='upper left')
axes[1,0].tick_params(axis='x', rotation=45)

#Top 10 teams all-time wins (updated with 2026)
wins = m_normal['winner'].value_counts().head(10)
colors = [TEAM_COLORS.get(t, '#999999') for t in wins.index]
axes[1,1].barh(wins.index[::-1], wins.values[::-1], color=colors[::-1])
axes[1,1].set_title("All-Time Top 10 Teams by Wins (2008–2026)")
axes[1,1].set_xlabel("Total Wins")

#2026 team win counts
wins_26 = m_normal[m_normal['season'] == 2026]['winner'].value_counts()
colors_26 = [TEAM_COLORS.get(SHORT_TO_FULL.get(t, t), '#999999') for t in wins_26.index]
axes[1,2].bar(wins_26.index, wins_26.values, color=colors_26, alpha=0.88)
axes[1,2].set_title("IPL 2026 — Wins per Team (so far)")
axes[1,2].set_xlabel("Team"); axes[1,2].set_ylabel("Wins")
for x, y in zip(wins_26.index, wins_26.values):
    axes[1,2].text(x, y+0.1, str(y), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_2008_2026.png'), bbox_inches='tight', dpi=120)
plt.show()

## Section 7 — Feature Engineering on Full 2008–2026 Dataset

In [ ]:
m = matches_all[matches_all['result'] == 'normal'].copy()
m = m[m['winner'].notna()].copy()
m = m.sort_values('date').reset_index(drop=True)

m['run_rate_1']     = m['first_innings_score'] / m['first_innings_overs'].replace(0, np.nan)
m['run_rate_2']     = m['second_innings_score'] / m['second_innings_overs'].replace(0, np.nan)
m['score_diff']     = m['second_innings_score'] - m['first_innings_score']
m['wkts_in_hand']   = 10 - m['second_innings_wickets']
m['is_day_night_i'] = m['is_day_night'].astype(int)

def rolling_team_features(df):
    rows = []
    for idx, row in df.iterrows():
        t1, t2, venue = row['team1'], row['team2'], row['venue']
        season = row['season']
        past = df.loc[:idx-1]

        # H2H win rate for team1
        h2h = past[((past['team1']==t1)&(past['team2']==t2))|
                   ((past['team1']==t2)&(past['team2']==t1))]
        h2h_rate = (h2h['winner']==t1).sum() / max(len(h2h), 1)

        # Team1 venue win rate
        v1 = past[((past['team1']==t1)|(past['team2']==t1)) & (past['venue']==venue)]
        v1_rate = (v1['winner']==t1).sum() / max(len(v1), 1)

        # Last 5 form
        t1m = past[(past['team1']==t1)|(past['team2']==t1)].tail(5)
        t2m = past[(past['team1']==t2)|(past['team2']==t2)].tail(5)
        t1_form = (t1m['winner']==t1).sum() / max(len(t1m), 1)
        t2_form = (t2m['winner']==t2).sum() / max(len(t2m), 1)

        # Season win rate
        sp  = past[past['season']==season]
        t1s = sp[(sp['team1']==t1)|(sp['team2']==t1)]
        t2s = sp[(sp['team1']==t2)|(sp['team2']==t2)]
        t1_swr = (t1s['winner']==t1).sum() / max(len(t1s), 1)
        t2_swr = (t2s['winner']==t2).sum() / max(len(t2s), 1)

        rows.append([h2h_rate, v1_rate, t1_form, t2_form, t1_swr, t2_swr])

    return pd.DataFrame(rows, columns=['h2h_rate','venue_wr','t1_form','t2_form','t1_swr','t2_swr'])


print(" Computing rolling features for all 2008–2026 matches (~30 sec)...")
hist_feats = rolling_team_features(m)
m = pd.concat([m.reset_index(drop=True), hist_feats], axis=1)

# Target variable: 1 = chasing team won (batting second won)
m['target'] = (m['win_by'] == 'wickets').astype(int)

FEATURES_FULL = ['first_innings_score','first_innings_wickets','run_rate_1',
                 'second_innings_score','second_innings_wickets','score_diff',
                 'wkts_in_hand','is_day_night_i',
                 'h2h_rate','venue_wr','t1_form','t2_form','t1_swr','t2_swr']

FEATURES_PRE  = ['h2h_rate','venue_wr','t1_form','t2_form','t1_swr','t2_swr','is_day_night_i']

print(f" Features computed for {len(m):,} matches (2008–2026)")
print(f"   Season 2026 rows: {(m['season']==2026).sum()}")
m[FEATURES_PRE + ['season','winner']].tail(5)

## Section 8 — Train Model A: 2008–2025 Only (Baseline)

In [ ]:
m_train_only = m[m['season'] < 2026].copy()
m_test_2026  = m[m['season'] == 2026].copy()

X_train_full = m_train_only[FEATURES_FULL].fillna(0)
X_test_2026  = m_test_2026[FEATURES_FULL].fillna(0)
y_train      = m_train_only['target']
y_test_2026  = m_test_2026['target']

X_train_pre  = m_train_only[FEATURES_PRE].fillna(0)
X_test_2026p = m_test_2026[FEATURES_PRE].fillna(0)

#  train/test split for historical eval 
X_tr, X_te, y_tr, y_te = train_test_split(X_train_full, y_train, test_size=0.2, random_state=42)
X_tr_p, X_te_p, y_tr_p, y_te_p = train_test_split(X_train_pre, y_train, test_size=0.2, random_state=42)

scaler_A_full = StandardScaler()
scaler_A_pre  = StandardScaler()

clf_A_full = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
clf_A_pre  = LogisticRegression(max_iter=1000, C=0.5, random_state=42)

clf_A_full.fit(scaler_A_full.fit_transform(X_tr), y_tr)
clf_A_pre.fit(scaler_A_pre.fit_transform(X_tr_p), y_tr_p)

acc_A_full_hist  = accuracy_score(y_te,   clf_A_full.predict(scaler_A_full.transform(X_te)))
acc_A_pre_hist   = accuracy_score(y_te_p, clf_A_pre.predict(scaler_A_pre.transform(X_te_p)))
acc_A_full_2026  = accuracy_score(y_test_2026,  clf_A_full.predict(scaler_A_full.transform(X_test_2026)))
acc_A_pre_2026   = accuracy_score(y_test_2026,  clf_A_pre.predict(scaler_A_pre.transform(X_test_2026p)))

print("="*55)
print("  MODEL A — Trained on 2008-2025 ONLY")
print("="*55)
print(f"  Full-match (2008-25 test split) : {acc_A_full_hist*100:.2f}%")
print(f"  Pre-match  (2008-25 test split) : {acc_A_pre_hist*100:.2f}%")
print(f"  Full-match on 2026 (unseen)     : {acc_A_full_2026*100:.2f}%")
print(f"  Pre-match  on 2026 (unseen)     : {acc_A_pre_2026*100:.2f}%")
print("="*55)

## Section 9 — Train Model B: Full 2008–2026 Dataset

In [ ]:
# ── Train on ALL data: 2008–2026 ─────────────────────────────────────────────
X_all_full = m[FEATURES_FULL].fillna(0)
X_all_pre  = m[FEATURES_PRE].fillna(0)
y_all      = m['target']

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_all_full, y_all, test_size=0.2, random_state=42)
X_tr2p, X_te2p, y_tr2p, y_te2p = train_test_split(X_all_pre, y_all, test_size=0.2, random_state=42)

scaler_B_full = StandardScaler()
scaler_B_pre  = StandardScaler()

clf_B_full = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
clf_B_pre  = LogisticRegression(max_iter=1000, C=0.5, random_state=42)

clf_B_full.fit(scaler_B_full.fit_transform(X_tr2),  y_tr2)
clf_B_pre.fit(scaler_B_pre.fit_transform(X_tr2p), y_tr2p)

acc_B_full = accuracy_score(y_te2,  clf_B_full.predict(scaler_B_full.transform(X_te2)))
acc_B_pre  = accuracy_score(y_te2p, clf_B_pre.predict(scaler_B_pre.transform(X_te2p)))
cv_B       = cross_val_score(clf_B_full, scaler_B_full.transform(X_tr2), y_tr2, cv=5).mean()

print("="*55)
print("  MODEL B — Trained on 2008-2026 (FULL)")
print("="*55)
print(f"  Full-match accuracy : {acc_B_full*100:.2f}%")
print(f"  Pre-match accuracy  : {acc_B_pre*100:.2f}%")
print(f"  5-fold CV score     : {cv_B*100:.2f}%")
print("="*55)

# Save Model B (the best one for prediction)
joblib.dump(clf_B_full,    os.path.join(SAVE_DIR, 'model_B_clf_full.pkl'))
joblib.dump(clf_B_pre,     os.path.join(SAVE_DIR, 'model_B_clf_pre.pkl'))
joblib.dump(scaler_B_full, os.path.join(SAVE_DIR, 'scaler_B_full.pkl'))
joblib.dump(scaler_B_pre,  os.path.join(SAVE_DIR, 'scaler_B_pre.pkl'))
joblib.dump(m,             os.path.join(SAVE_DIR, 'processed_matches_2008_2026.pkl'))

## Section 10 — Side-by-Side Model Comparison

In [ ]:
# ── Comparison bar chart ──────────────────────────────────────────────────────
comparison = {
    'Model A\nFull-match\n(2008-25 split)'  : acc_A_full_hist,
    'Model A\nPre-match\n(2008-25 split)'   : acc_A_pre_hist,
    'Model A\nFull-match\non 2026 (unseen)' : acc_A_full_2026,
    'Model A\nPre-match\non 2026 (unseen)'  : acc_A_pre_2026,
    'Model B\nFull-match\n(2008-26 split)'  : acc_B_full,
    'Model B\nPre-match\n(2008-26 split)'   : acc_B_pre,
}

colors = ['#004BA0','#F7A721','#EC1C24','#FF6600','#1C4899','#00A3E0']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Model A (2008–2025) vs Model B (2008–2026) — Accuracy Comparison',
             fontsize=13, fontweight='bold')

# Left: bar chart
bars = axes[0].bar(comparison.keys(), [v*100 for v in comparison.values()],
                   color=colors, alpha=0.88, edgecolor='white', linewidth=1.2)
axes[0].set_ylabel('Accuracy (%)', fontsize=11)
axes[0].set_ylim(40, 100)
axes[0].axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='Random (50%)')
axes[0].legend()
for bar, val in zip(bars, comparison.values()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val*100:.1f}%', ha='center', fontsize=9, fontweight='bold')
axes[0].tick_params(axis='x', labelsize=8)

# Right: confusion matrices
cm_A = confusion_matrix(y_test_2026, clf_A_full.predict(scaler_A_full.transform(X_test_2026)))
cm_B_data = m[m['season'] == 2026][FEATURES_FULL].fillna(0)
cm_B_y    = m[m['season'] == 2026]['target']
cm_B = confusion_matrix(cm_B_y, clf_B_full.predict(scaler_B_full.transform(cm_B_data)))

ConfusionMatrixDisplay(cm_A, display_labels=['Bat 1st','Chase']).plot(
    ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title(f'Model A on 2026 — Acc: {acc_A_full_2026*100:.1f}%')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'model_comparison.png'), bbox_inches='tight', dpi=120)
plt.show()

# Drift analysis
drift = acc_A_full_2026 - acc_A_full_hist
print(f"\n Model A drift on 2026: {drift*100:+.2f}%")
if abs(drift) < 0.05:
    print(" Model is stable — generalizes well to 2026 data")
elif drift < 0:
    print("  Model degraded on 2026 — Model B (retrained with 2026) is recommended")
else:
    print(" Model actually improved on 2026!")

## Section 11 — Predict Upcoming Match (Uses Model B)

In [ ]:
def predict_upcoming_match(team1, team2, venue, is_day_night=True, verbose=True):
    """
    Predicts the winner of any upcoming IPL match.
    Uses Model B (trained on 2008–2026) with full rolling history.
    
    Parameters:
        team1        : Full name e.g. 'Mumbai Indians'
        team2        : Full name e.g. 'Chennai Super Kings'
        venue        : Ground name (partial match OK) e.g. 'Wankhede'
        is_day_night : True for evening matches (default True)
    """
    completed = m[m['result'] == 'normal'].copy()

    # H2H rate (team1 win %)
    h2h = completed[
        ((completed['team1']==team1) & (completed['team2']==team2)) |
        ((completed['team1']==team2) & (completed['team2']==team1))
    ]
    h2h_rate = (h2h['winner']==team1).sum() / max(len(h2h), 1)

    # Venue win rate for team1
    v_keyword = venue.split(',')[0].strip()
    v1 = completed[
        ((completed['team1']==team1) | (completed['team2']==team1)) &
        completed['venue'].str.contains(v_keyword, case=False, na=False)
    ]
    venue_wr = (v1['winner']==team1).sum() / max(len(v1), 1)

    # Last 5 form
    t1m = completed[(completed['team1']==team1)|(completed['team2']==team1)].tail(5)
    t2m = completed[(completed['team1']==team2)|(completed['team2']==team2)].tail(5)
    t1_form = (t1m['winner']==team1).sum() / max(len(t1m), 1)
    t2_form = (t2m['winner']==team2).sum() / max(len(t2m), 1)

    # 2026 season win rate
    sp  = completed[completed['season']==2026]
    t1s = sp[(sp['team1']==team1)|(sp['team2']==team1)]
    t2s = sp[(sp['team1']==team2)|(sp['team2']==team2)]
    t1_swr = (t1s['winner']==team1).sum() / max(len(t1s), 1)
    t2_swr = (t2s['winner']==team2).sum() / max(len(t2s), 1)

    features = np.array([[h2h_rate, venue_wr, t1_form, t2_form, t1_swr, t2_swr, int(is_day_night)]])
    features_scaled = scaler_B_pre.transform(features)
    proba = clf_B_pre.predict_proba(features_scaled)[0]

    t1_win_prob = proba[0]
    t2_win_prob = proba[1]

    t1_short = FULL_TO_SHORT.get(team1, team1)
    t2_short = FULL_TO_SHORT.get(team2, team2)

    if verbose:
        winner = t1_short if t1_win_prob > t2_win_prob else t2_short
        conf   = max(t1_win_prob, t2_win_prob)
        bar1   = '' * int(t1_win_prob * 20)
        bar2   = '' * int(t2_win_prob * 20)

        print('\n' + '='*58)
        print(f'   MATCH PREDICTION  (Model B — 2008 to 2026)')
        print(f'  {team1}  vs  {team2}')
        print(f'   Venue: {venue}')
        print('='*58)
        print(f'   Features used:')
        print(f'     H2H win rate ({t1_short})    : {h2h_rate*100:.1f}% ({len(h2h)} matches)')
        print(f'     Venue win rate ({t1_short})  : {venue_wr*100:.1f}% ({len(v1)} matches)')
        print(f'     Last 5 form  {t1_short}/{t2_short}: {t1_form*100:.0f}% / {t2_form*100:.0f}%')
        print(f'     2026 form    {t1_short}/{t2_short}: {t1_swr*100:.0f}% / {t2_swr*100:.0f}%')
        print('-'*58)
        print(f'   Predicted Winner : {winner}')
        print(f'   Confidence       : {conf*100:.1f}%')
        print(f'   Win Probabilities:')
        print(f'     {t1_short:<6}: {t1_win_prob*100:5.1f}%  {bar1}')
        print(f'     {t2_short:<6}: {t2_win_prob*100:5.1f}%  {bar2}')
        print('='*58)

    return {team1: round(t1_win_prob, 4), team2: round(t2_win_prob, 4)}


print(" Prediction function ready!")
print("\nAvailable full team names:")
for full in sorted(TEAM_COLORS.keys()):
    print(f"  '{full}'")

## Section 12 — Make Predictions

In [ ]:
# ── Predict any match — edit these ───────────────────────────────────────────
predict_upcoming_match(
    team1        = 'Mumbai Indians',
    team2        = 'Chennai Super Kings',
    venue        = 'Wankhede Stadium, Mumbai',
    is_day_night = True
)

In [ ]:
# ── Semi-final probability matrix ─────────────────────────────────────────────
# Update TOP4 with actual top 4 teams after league phase
TOP4 = [
    'Punjab Kings',
    'Royal Challengers Bangalore',
    'Sunrisers Hyderabad',
    'Rajasthan Royals',
]
PLAYOFF_VENUE = 'Narendra Modi Stadium, Ahmedabad'

print('\n PLAYOFF WIN PROBABILITY MATRIX (Model B — 2008–2026)')
print('='*80)
header = f'{"":28}' + ''.join(f'{FULL_TO_SHORT.get(t,t):>10}' for t in TOP4)
print(header)
print('-'*75)

for t1 in TOP4:
    row_str = f'{FULL_TO_SHORT.get(t1,t1):<28}'
    for t2 in TOP4:
        if t1 == t2:
            row_str += f'{"---":>10}'
        else:
            result = predict_upcoming_match(t1, t2, PLAYOFF_VENUE, verbose=False)
            row_str += f'{result[t1]*100:>9.1f}%'
    print(row_str)

print('='*80)
print('(Each cell = row team win % against column team)')